In [2]:
import pandas as pd
import numpy as np
import importlib
import statsmodels.formula.api as smf

from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.cov_struct import Autoregressive, Unstructured

import analysis
importlib.reload(analysis)

<module 'analysis' from 'c:\\Users\\JoleneWium\\Documents\\Projects\\biostats\\PPH7095F\\20260402\\code\\analysis.py'>

In [3]:
df = analysis.load_data()

In [4]:
dat = df[
    [
        "id",
        "obs_time",
        "satisfaction",
        "sex",
        "age_marriage",
        "cohab",
        "income",
        "hw_all",
    ]
].dropna().copy()

dat = dat.sort_values(["id", "obs_time"]).reset_index(drop=True)

# visit order within person
dat["time_index"] = dat.groupby("id").cumcount().astype(int)

formula = (
    "satisfaction ~ time_index + sex + age_marriage + "
    "cohab + income + hw_all"
)

In [5]:
# LME 1: Random intercept only
lme1 = smf.mixedlm(
    formula=formula,
    data=dat,
    groups=dat["id"],   # random intercept
).fit()
lme1.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
          Mixed Linear Model Regression Results
==========================================================
Model:            MixedLM Dependent Variable: satisfaction
No. Observations: 3813    Method:             REML        
No. Groups:       494     Scale:              0.8455      
Min. group size:  3       Log-Likelihood:     -5786.3216  
Max. group size:  17      Converged:          Yes         
Mean group size:  7.7                                     
----------------------------------------------------------
               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept       6.968    0.425  16.397 0.000  6.135  7.801
sex[T.male]    -0.434    0.153  -2.837 0.005 -0.734 -0.134
time_index     -0.308    0.006 -55.016 0.000 -0.319 -0.297
age_marriage    0.008    0.013   0.648 0.517 -0.017  0.033
cohab           0.262    0.178   1.473 0.141 -0.087  0.612
income         -0.000    0.000  -0.384 0.701 -0.000  0.000
hw_all         -0.579    0.152  -3.815 0.000 -0.876 -0.281
Group Var       1.693    0.136                            
==========================================================

"""

In [7]:
# LME 2: Random intercept + random slope for time
lme2 = smf.mixedlm(
    formula=formula,
    data=dat,
    groups=dat["id"],
    re_formula="~time_index",  # adds random slope
).fit()
lme2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:                MixedLM   Dependent Variable:   satisfaction
No. Observations:     3813      Method:               REML        
No. Groups:           494       Scale:                0.7607      
Min. group size:      3         Log-Likelihood:       -5738.6336  
Max. group size:      17        Converged:            Yes         
Mean group size:      7.7                                         
------------------------------------------------------------------
                       Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               6.920    0.427  16.221 0.000  6.084  7.756
sex[T.male]            -0.455    0.153  -2.970 0.003 -0.755 -0.155
time_index             -0.342    0.009 -39.183 0.000 -0.359 -0.325
age_marriage            0.011    0.013   0.819 0.413 -0.015  0.036
cohab                   0.298    0.179   1.669 0.095 -0.052  0.648
income                 -0.000    0.000  -0.217 0.829 -0.000  0.000
hw_all                 -0.542    0.152  -3.567 0.000 -0.840 -0.244
Group Var               1.659    0.155                            
Group x time_index Cov -0.006    0.014                            
time_index Var          0.012    0.003                            
==================================================================

"""